# ECCOFS `avg`: vertical temperature section off New Jersey with xroms

Opens the whole-column best-estimate virtual Icechunk store (`eccofs-avg-best-nosub`) on [Source Cooperative](https://source.coop), cuts out the mid-Atlantic Bight, and uses [xroms](https://xroms.readthedocs.io) to build the grid and the terrain-following depths (`z_rho`) and to interpolate along an offshore transect from the New Jersey coast across the shelf break (`xroms.interpll`). The section is drawn with hvPlot.

A section reads every vertical level in the box, so this notebook uses the `-nosub` store, where all 50 levels of a tile are one chunk. It reads the section data about 2x faster than the default `eccofs-avg-best` store, whose chunks are single levels and are better for maps and point time series (see the README, "Chunking").

No credentials are needed. It needs icechunk 2.x (Python 3.12+), `xroms` 0.6.2 with `xgcm` 0.8.1 (newer xgcm releases break xroms' grid parsing) and `xesmf`, which `xroms.interpll` wraps.

In [ ]:
# %pip install -q "xroms==0.6.2" "xgcm==0.8.1" xesmf hvplot datashader geoviews cartopy icechunk zarr

In [ ]:
import cartopy.crs as ccrs
import geoviews as gv
import holoviews as hv
import hvplot.xarray  # noqa: F401
import icechunk
import numpy as np
import xarray as xr
import xroms
import zarr

zarr.config.set({"async.concurrency": 32})

# whole-column chunks (all 50 levels of a tile): the better layout for sections. For maps and time series use eccofs-avg-best
URL = "https://data.source.coop/ocean-icechunks/noaa-eccofs/eccofs-avg-best-nosub"
SOURCE = "s3://noaa-nos-eccofs-pds/"   # public bucket holding the NetCDF bytes

# transect: off Atlantic City, NJ, running south-east across the shelf and slope (edit me)
START = (-74.30, 39.30)   # lon, lat
END = (-72.00, 37.20)
WHEN = "2026-11-15 12:00"

In [ ]:
# the NetCDF bytes live in NOAA's bucket, outside the store: authorize that virtual chunk container (anonymous)
repo = icechunk.Repository.open(
    icechunk.http_storage(URL),
    authorize_virtual_chunk_access=icechunk.containers_credentials({SOURCE: icechunk.s3_anonymous_credentials()}),
)
ds = xr.open_zarr(repo.readonly_session("main").store, consolidated=False, chunks={})
print(f"{ds.sizes['time']} times: {ds.time.values[0]} to {ds.time.values[-1]}")

## Cut out the region and one time

The full grid is 1443 x 1667 x 50 levels, so subset first. The rho rows and columns that touch the box become `X`/`Y` slices for `xroms.subset`, which slices the staggered `u`/`v`/`psi` dimensions consistently (one fewer point than `rho` across the staggering), as xgcm requires. It only touches dimensions that have a coordinate, and the store's horizontal dimensions have none, so index coordinates are added first.

In [ ]:
box = ((ds.lon_rho > -76) & (ds.lon_rho < -70) & (ds.lat_rho > 36) & (ds.lat_rho < 41)).load()
rows, cols = np.flatnonzero(box.any("xi_rho")), np.flatnonzero(box.any("eta_rho"))   # rho rows / columns that touch the box

# xroms.subset slices every staggered grid consistently, but only for dimensions that have a coordinate
hdims = [d for d in ds.dims if d.startswith(("xi_", "eta_"))]
region = xroms.subset(
    ds.assign_coords({d: np.arange(ds.sizes[d]) for d in hdims}),
    X=slice(cols[0], cols[-1] + 1), Y=slice(rows[0], rows[-1] + 1),
)
fields = ["temp", "u", "v", "zeta", "h", "Cs_r", "Cs_w", "pm", "pn", "angle", "f", "mask_rho", "mask_u", "mask_v", "mask_psi"]
scalars = ["Vtransform", "Vstretching", "theta_s", "theta_b", "Tcline", "hc", "rho0", "spherical"]
sub = region[fields + scalars].sel(time=WHEN, method="nearest")
print("time:", sub.time.values, "| subset:", dict(sub.sizes))

## xroms dataset and depths

xroms builds `z_rho` from `Vtransform`, `Vstretching`, `theta_s`, `theta_b` and `hc`, which are stored with the data (added back to the store by `scripts/add_scalars.py`). `roms_dataset` builds the xgcm grid and the depth fields. Two things the ECCOFS store needs: the SGRID `Conventions` attribute is dropped (xroms builds its own grid from the dimension names) and the arrays are rechunked to single chunks along the horizontal dimensions, which xgcm and `interpll` require.

In [ ]:
sub.attrs["Conventions"] = "CF-1.4"
sub = sub.chunk({d: -1 for d in sub.dims})
rds, xgrid = xroms.roms_dataset(sub)
z_rho = rds.z_rho
assert float(z_rho.isel(s_rho=-1).max()) < 5 and float(z_rho.isel(s_rho=0).min()) < -3000, "unexpected depths"
print(f"z_rho: surface {float(z_rho.isel(s_rho=-1).mean()):.2f} m mean, deepest {float(z_rho.isel(s_rho=0).min()):.0f} m")

## Transect

Points are spaced about 3 km (the grid size) along the line between `START` and `END`. `xroms.interpll(..., which="pairs")` regrids bilinearly (with xESMF, on the sphere) to those lon/lat pairs and also interpolates `z_rho`, so the curvilinear grid needs no projection. Land is masked first.

In [ ]:
def haversine_km(lon1, lat1, lon2, lat2):
    p = np.pi / 180
    a = np.sin((lat2 - lat1) * p / 2) ** 2 + np.cos(lat1 * p) * np.cos(lat2 * p) * np.sin((lon2 - lon1) * p / 2) ** 2
    return 12742 * np.arcsin(np.sqrt(a))

total_km = haversine_km(*START, *END)
n = int(total_km / 3) + 1
lons = np.linspace(START[0], END[0], n)
lats = np.linspace(START[1], END[1], n)
dist = np.linspace(0, total_km, n)
print(f"{n} points over {total_km:.0f} km")

In [ ]:
%%time
mask = rds.mask_rho == 1
temp = rds.temp.where(mask).load()
h = rds.h.where(mask).load()

sec = xroms.interpll(temp, lons, lats, which="pairs")      # (s_rho, locations) with z_rho along the section
sec_h = xroms.interpll(h, lons, lats, which="pairs")
sec

## Section

Temperature is drawn on the true (moving) depth of each level, `z_rho`, with the sea floor from `h`. Look for the shelf water inshore of the shelf break, the thermocline sloping down across the slope, and the warm Gulf Stream-influenced water offshore.

In [ ]:
# quadmesh wants x and y on the same dimensions: give the distance as a 2-D coordinate like z_rho
sec = sec.assign_coords(km=(("s_rho", "locations"), np.broadcast_to(dist, sec.z_rho.shape).copy()))
sec.attrs.update(long_name="potential temperature", units="degC")

section = sec.hvplot.quadmesh(
    x="km", y="z_rho", rasterize=True, cmap="turbo", colorbar=True,
    xlabel="distance from the coast (km)", ylabel="depth (m)",
    title=f"ECCOFS temperature, {np.datetime_as_string(sub.time.values, unit='h')} UTC",
    frame_width=700, frame_height=350, ylim=(-1000, 5), xlim=(0, float(dist[-1])),
)
floor = hv.Area((dist, -sec_h.values, np.full(n, -4000.0)), kdims="km", vdims=["bottom", "floor"]).opts(color="#4d4d4d", line_width=0)
section * floor

In [ ]:
sst = temp.isel(s_rho=-1).rename("sst")
surface = sst.hvplot.quadmesh(
    x="lon_rho", y="lat_rho", rasterize=True, geo=True, tiles="OSM", cmap="turbo",
    frame_width=450, title="Surface temperature (degC) and transect",
)
# geoviews path with a CRS, so the degrees are projected like the map (a plain hv.Path would be read as metres, near 0,0)
track = np.c_[lons, lats]
line = gv.Path([track], crs=ccrs.PlateCarree()).opts(color="white", line_width=3) * gv.Path([track], crs=ccrs.PlateCarree()).opts(color="black", line_width=1)
surface * line

Depths down to 1000 m are shown so the shelf is readable; use `ylim=(-3500, 5)` to see the full slope.